# PySpark Setup on Google Colab
## Hadoop & Spark - ISEN 2025/2026
---

## Prerequisites

For this Lab you will need:
- Ideally, prior Python knowledge
- A free Google Account (For Google Drive and Google Colab)
- Space left on your Google Drive Account (First 15Go are free)
- A free Kaggle Account
- A free NGROK Account

---
## Configuration

Set your NGROK and Kaggle tokens below:

In [ ]:
# NGROK Authentication Token
NGROK_TOKEN = ""

# Kaggle API Token
KAGGLE_API_TOKEN = ""

---
## 1. Install Dependencies

Install the following dependencies (may take up to 5 minutes in Colab):
- **Java 8** - Required runtime for Spark
- **Apache Spark** with Hadoop
- **Findspark** - Used to locate Spark in the system

In [2]:
# Install Java 8
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Download Apache Spark 3.5.8 with Hadoop 3
!wget -q http://archive.apache.org/dist/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz

# Extract Spark
!tar xf spark-3.5.8-bin-hadoop3.tgz

# Install findspark
!pip install -q findspark

---
## 2. Set Environment Variables

Configure the paths for Java and Spark.

In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

---
## 3. Initialize Findspark

Import and initialize findspark to locate Spark installation.

In [4]:
import findspark
findspark.init()

---
## 4. Create SparkSession

Import PySpark and create a SparkSession.

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CollabApp") \
    .master("local[*]") \
    .config('spark.ui.port', '4050') \
    .getOrCreate()

# Enable eager evaluation for better display in notebooks
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [6]:
# Display Spark session info
spark

---
## 5. Show Spark UI using NGROK

### What is NGROK?
NGROK is a unified ingress platform. This will allow us to redirect the local URLs of the Google Colab Notebooks to a URL we can access.

### Important
You need to create a NGROK account: https://ngrok.com/

You will need to:
1. Register on NGROK website
2. Retrieve your auth token
3. Verify your email address before launching the code

You will find the auth token in your NGROK dashboard once logged in.

### Install NGROK

In [7]:
# Install ngrok
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com buster main" \
  | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt update \
  && sudo apt install ngrok

deb https://ngrok-agent.s3.amazonaws.com buster main
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://ngrok-agent.s3.amazonaws.com buster InRelease [20.3 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ngrok-agent.s3.amazonaws.com buster/main amd64 Packages [10.8 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ub

### Set NGROK Authentication Token and Start Tunnel

In [8]:
# Set authentication token (using the NGROK_TOKEN defined at the beginning)
get_ipython().system_raw(f'ngrok config add-authtoken {NGROK_TOKEN}')

# Start ngrok tunnel to Spark UI
get_ipython().system_raw('ngrok http http://localhost:4050 &')

### Get the Public URL to Access Spark UI

In [9]:
# Get the public URL for Spark UI
!curl -s http://localhost:4040/api/tunnels | grep -Po 'public_url":"(?=https)\K[^"]*'

---
## 6. Download & Load a DataFrame from Kaggle

### What is Kaggle?
Kaggle is a website where you can find free datasets and data competitions with a large datascience community.

### Install Kaggle CLI

In [10]:
# Install Kaggle CLI
!pip install -q kaggle

### Configure Kaggle API Token

In [11]:
import os

# Set Kaggle API Token environment variable
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

# Alternative method: Create kaggle.json file
!mkdir -p ~/.kaggle
import json
kaggle_config = {"username": "your_username", "key": KAGGLE_API_TOKEN}
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_config, f)
!chmod 600 ~/.kaggle/kaggle.json

### Download Dataset from Kaggle

Download the LEGO Sets and Themes Database dataset:
- Dataset link: https://www.kaggle.com/datasets/jkraak/lego-sets-and-themes-database

In [12]:
# Download the LEGO dataset from Kaggle
!kaggle datasets download -d jkraak/lego-sets-and-themes-database

# Unzip the downloaded file
!unzip lego-sets-and-themes-database.zip

Dataset URL: https://www.kaggle.com/datasets/jkraak/lego-sets-and-themes-database
License(s): CC0-1.0
  0% 0.00/374k [00:00<?, ?B/s]
100% 374k/374k [00:00<00:00, 565MB/s]
Archive:  lego-sets-and-themes-database.zip
  inflating: lego_sets_and_themes.csv  


### Load DataFrame with PySpark

With Spark imported previously, you only have to read it now.

> **Note:** PySpark commands are often very similar to pandas!

In [13]:
# Read CSV file with PySpark
df = spark.read.csv('lego_sets_and_themes.csv', header=True, sep=",")
df.show(5)

+----------+--------------------+-------------+---------------+--------------------+----------+
|set_number|            set_name|year_released|number_of_parts|           image_url|theme_name|
+----------+--------------------+-------------+---------------+--------------------+----------+
|     001-1|               Gears|       1965.0|           43.0|https://cdn.rebri...|   Technic|
|     002-1|4.5V Samsonite Ge...|       1965.0|            3.0|https://cdn.rebri...|   Technic|
|    1030-1|TECHNIC I: Simple...|       1985.0|          210.0|https://cdn.rebri...|   Technic|
|    1038-1|  ERBIE the Robo-Car|       1985.0|          120.0|https://cdn.rebri...|   Technic|
|    1039-1|Manual Control Set 1|       1986.0|           39.0|https://cdn.rebri...|   Technic|
+----------+--------------------+-------------+---------------+--------------------+----------+
only showing top 5 rows



---
## 7. Prepare Data for Later

We cast 2 columns and set the delimiter as ';' to make our life easier later on:

In [14]:
from pyspark.sql.types import *

# Cast columns to appropriate types
df = df.withColumn("number_of_parts", df["number_of_parts"].cast(IntegerType()))
df = df.withColumn("year_released", df["year_released"].cast(IntegerType()))

# Write cleaned data with semicolon delimiter
df.write.options(delimiter=';').mode("overwrite").csv("lego_sets_and_themes_clean.csv")

---
## Setup Complete!

You can now use PySpark in Google Colab. The Spark UI is accessible via the NGROK URL displayed above.

### Quick Test

In [15]:
# Quick test - Create a simple DataFrame
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["Name", "Age"]

df = spark.createDataFrame(data, columns)
df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



In [16]:
# Stop Spark session when done
# spark.stop()

---
# Lab: RDDs
## Hadoop & Spark - ISEN 2025/2026

---

## Create a SparkContext

To manipulate RDDs, we need a SparkContext. We can get it from our SparkSession.

In [ ]:
# Create SparkContext from SparkSession
sc = spark.sparkContext
sc

---
## 1. Create a RDD using `parallelize`

The `parallelize` method creates an RDD from a local collection.

In [ ]:
# Create an RDD using parallelize
rddParallel = sc.parallelize(
    [
        [1, "one"],
        [2, "two"],
        [3, "three"],
        [4, "four"],
        [5, "five"],
    ]
)
rddParallel

### Check RDD partitions

In [ ]:
# Get the number of partitions
rddParallel.getNumPartitions()

### Get complete result from RDD

In [ ]:
# Collect all elements from the RDD
output = rddParallel.collect()
print(output)

### Get X records from the RDD

In [ ]:
# Get 3 records from the RDD
rddParallel.take(3)

### Get the first element

In [ ]:
# Get the first element
rddParallel.first()

### `first()` vs `take(1)`

Please note that in the above command you had an array as output. Here you have a single element, which happens to be an array.

> **Important:** `first()` ≠ `take(1)`

In [ ]:
# Compare first() with take(1)
print("first():", rddParallel.first())
print("take(1):", rddParallel.take(1))

---
## 2. Create an RDD reading a file

We will delete the header as an RDD is just a collection of elements. Note that every line is of type string.

In [ ]:
# Note: NOT the clean ONE!
rddFile = sc.textFile('lego_sets_and_themes.csv')
rddFile.take(5)

In [ ]:
# Remove the header row
header = rddFile.first()
rddFile = rddFile.filter(lambda row: row != header)
rddFile.take(5)

### See the number of partitions

In [ ]:
# Get the number of partitions
rddFile.getNumPartitions()

### Control the number of partitions

You can control the number of partitions by specifying at read time. Here we use 4 partitions:

In [ ]:
# Note: The clean one!
rddFile = sc.textFile('lego_sets_and_themes_clean.csv', 4)

# Remove the first row again
header = rddFile.first()
rddFile = rddFile.filter(lambda row: row != header)

# Check number of partitions
rddFile.getNumPartitions()

In [ ]:
# View some data from the clean file
rddFile.take(5)